<a href="https://colab.research.google.com/github/Venu-max/NASSCOM-AI/blob/main/Day9_U19_%E2%80%94_Data_Labeling_%26_Data_Centric_AI_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Real-world brief: cleaning noisy inspection labels on an aluminium casting line
A foundry classifies castings as defect or ok. Labels come from human inspectors who disagree, especially on borderline parts — so the labels you train on contain errors. In the data-centric philosophy, improving these labels often beats swapping in a fancier model. In this lab you'll measure inter-annotator agreement, build consensus labels, detect likely label errors, run an active-learning loop, and try weak supervision — then show that fixing the data lifts performance.

Resource provided: casting_inspection.csv — objective measurements, three inspector labels (inspector_A/B/C), the noisy label_recorded you'd normally train on, and a hidden true_defect ground truth (provided here only so you can measure progress). Keep it beside this notebook.

Phase E — Data-Centric AI.

objectives
Quantify inter-annotator agreement with Cohen's kappa

Build consensus (majority-vote) labels and compare to ground truth

Detect likely mislabelled rows with confident-learning-style logic

Show that cleaning labels improves a model (data-centric win)

Run an active-learning loop and combine weak-supervision rules

In [1]:
# === SETUP: load the provided file (regenerate it if missing) ===
import os
import numpy as np
import pandas as pd


def build_castings(csv_path="casting_inspection.csv", seed=190, verbose=False):
    """Aluminium casting inspection records for a DATA-CENTRIC AI lab (U19).

    Each casting has objective process/measurement features and a true (latent) defect
    state. Three human inspectors each label it — but humans are noisy and disagree, more
    so on borderline parts. The column you'd actually TRAIN on (`label_recorded`) is a
    single inspector's call and therefore contains real labelling errors.

    Columns:
      porosity_pct, wall_thickness_mm, fill_time_s, melt_temp_c, pressure_bar, surface_ra_um
                                          -> objective features
      inspector_A / inspector_B / inspector_C  -> three human labels (0 ok / 1 defect)
      label_recorded                      -> the noisy single-annotator label (train on this)
      true_defect                         -> hidden ground truth (for teaching/validation only)
    """
    rng = np.random.default_rng(seed)
    N = 2400

    porosity = rng.gamma(2.0, 1.1, N).clip(0, 12)
    wall = rng.normal(6.0, 0.8, N).clip(3.5, 8.5)
    fill_time = rng.normal(2.4, 0.5, N).clip(1.0, 4.5)
    melt_temp = rng.normal(710, 15, N).clip(660, 760)
    pressure = rng.normal(95, 12, N).clip(60, 130)
    surface_ra = rng.normal(3.2, 0.9, N).clip(1.0, 7.0)

    # true defect: a SHARP function of genuine drivers so it is learnable from features
    # (steep sigmoid -> probabilities pushed toward 0/1 -> high but not perfect ceiling)
    drive = (0.55 * porosity + 1.3 * np.maximum(4.8 - wall, 0)
             + 1.1 * np.maximum(fill_time - 2.9, 0) + 0.45 * np.maximum(surface_ra - 3.6, 0)
             + 0.04 * np.maximum(melt_temp - 720, 0))
    thr = np.quantile(drive, 0.80)            # ~20% defect rate
    p_true = 1 / (1 + np.exp(-2.2 * (drive - thr)))   # gain 2.2 -> sharp, ~10% label noise
    true_defect = (rng.random(N) < p_true).astype(int)

    # "difficulty": borderline parts (p near 0.5) are where inspectors disagree
    difficulty = 1 - np.abs(p_true - 0.5) * 2          # 0 easy .. 1 hard
    def inspector(skill):
        # flip the true label with prob rising on hard parts, lower for higher skill
        flip_p = (0.07 + 0.55 * difficulty) * (1.0 - skill)
        flips = rng.random(N) < flip_p
        return np.where(flips, 1 - true_defect, true_defect)

    insp_A = inspector(0.80)
    insp_B = inspector(0.66)
    insp_C = inspector(0.52)          # least reliable inspector

    # label_recorded = the noisy HISTORICAL label. It carries a real inspector VISUAL BIAS:
    # rough-looking but truly-OK parts were over-called as defects, and some smooth true
    # defects were missed -> a SYSTEMATIC error (not just random), which a model will learn.
    label_recorded = true_defect.copy()
    ra_hi = np.quantile(surface_ra, 0.60); ra_lo = np.quantile(surface_ra, 0.40)
    rough_ok = (true_defect == 0) & (surface_ra > ra_hi)
    label_recorded[rough_ok & (rng.random(N) < 0.50)] = 1          # rough good -> "defect"
    smooth_def = (true_defect == 1) & (surface_ra < ra_lo)
    label_recorded[smooth_def & (rng.random(N) < 0.50)] = 0        # smooth defect -> missed
    rand_flip = rng.random(N) < 0.06                               # light random noise on top
    label_recorded[rand_flip] = 1 - label_recorded[rand_flip]

    df = pd.DataFrame({
        "porosity_pct": porosity.round(2), "wall_thickness_mm": wall.round(2),
        "fill_time_s": fill_time.round(2), "melt_temp_c": melt_temp.round(1),
        "pressure_bar": pressure.round(1), "surface_ra_um": surface_ra.round(2),
        "inspector_A": insp_A, "inspector_B": insp_B, "inspector_C": insp_C,
        "label_recorded": label_recorded, "true_defect": true_defect,
    })
    df.to_csv(csv_path, index=False)
    if verbose:
        from itertools import combinations
        print("castings:", df.shape)
        print("true defect rate:", round(true_defect.mean(), 3))
        print("recorded-label error rate vs truth:", round((df.label_recorded != df.true_defect).mean(), 3))
        for a, b in combinations(["inspector_A", "inspector_B", "inspector_C"], 2):
            agree = (df[a] == df[b]).mean()
            print(f"  raw agreement {a[-1]}-{b[-1]}: {agree:.3f}")
        cons = (df[["inspector_A", "inspector_B", "inspector_C"]].sum(axis=1) >= 2).astype(int)
        print("  consensus(majority) error rate:", round((cons != df.true_defect).mean(), 3))
    return df

if not os.path.exists('casting_inspection.csv'):
    build_castings(); print('Generated dataset file.')
else:
    print('Found the provided dataset file.')


Generated dataset file.


In [2]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
df = pd.read_csv('casting_inspection.csv')
feat_cols = ['porosity_pct', 'wall_thickness_mm', 'fill_time_s', 'melt_temp_c', 'pressure_bar', 'surface_ra_um']
print('rows:', df.shape)
print('recorded-label defect rate:', round(df.label_recorded.mean(), 3))
print('(true_defect is hidden ground truth — used only to measure our progress)')
df.head(3)

rows: (2400, 11)
recorded-label defect rate: 0.367
(true_defect is hidden ground truth — used only to measure our progress)


,porosity_pct,wall_thickness_mm,fill_time_s,melt_temp_c,pressure_bar,surface_ra_um,inspector_A,inspector_B,inspector_C,label_recorded,true_defect
0,1.28,5.11,2.73,710.8,103.7,2.08,0,0,1,0,0
1,1.14,5.47,2.37,706.4,85.7,2.84,1,0,0,1,0
2,0.58,6.69,2.16,709.9,92.9,3.90,0,0,0,0,0


In [ ]:
#1. How much do the inspectors agree?

In [3]:
# -----------------------------------------------------------
# 🔹 1A. COHEN'S KAPPA BETWEEN EACH PAIR OF INSPECTORS
# -----------------------------------------------------------
from sklearn.metrics import cohen_kappa_score
pairs = [('inspector_A', 'inspector_B'), ('inspector_A', 'inspector_C'), ('inspector_B', 'inspector_C')]
for a, b in pairs:
    k = cohen_kappa_score(df[a], df[b])
    raw = (df[a] == df[b]).mean()
    print(f'{a[-1]}-{b[-1]}: raw agreement {raw:.3f} | Cohen kappa {k:.3f}')
print('\nKappa corrects raw agreement for chance. <0.4 weak, 0.4-0.6 moderate, 0.6-0.8 substantial.')

A-B: raw agreement 0.887 | Cohen kappa 0.723
A-C: raw agreement 0.855 | Cohen kappa 0.648
B-C: raw agreement 0.837 | Cohen kappa 0.608

Kappa corrects raw agreement for chance. <0.4 weak, 0.4-0.6 moderate, 0.6-0.8 substantial.


In [ ]:
#EXERCISE 1 — Who is the outlier inspector?
#Compare each inspector against the hidden true_defect with cohen_kappa_score (in practice you wouldn't have truth — here it's to confirm the method).
#In a comment, name the least reliable inspector and explain why low pairwise kappa is a red flag for label quality.

In [4]:
from sklearn.metrics import cohen_kappa_score

# 1. Compare each inspector with the hidden true labels
inspectors = ['inspector_A', 'inspector_B', 'inspector_C']

for inspector in inspectors:
    kappa = cohen_kappa_score(df['true_defect'], df[inspector])
    print(f"{inspector}: Cohen's Kappa = {kappa:.3f}")

# 2. Comment:
# The inspector with the lowest Cohen's Kappa score is the least reliable.
# A low Kappa means the inspector's labels disagree with the true labels
# more often than expected and show poor consistency.
#
# Low pairwise Kappa between inspectors is a red flag because inconsistent
# labels introduce noise into the dataset. Machine learning models trained
# on unreliable labels may learn incorrect patterns, resulting in lower
# prediction accuracy and poor generalization.

inspector_A: Cohen's Kappa = 0.886
inspector_B: Cohen's Kappa = 0.806
inspector_C: Cohen's Kappa = 0.725


In [ ]:
#2. Consensus labels beat any single annotator

In [5]:

# -----------------------------------------------------------
# 🔹 2A. MAJORITY VOTE ACROSS THE THREE INSPECTORS
# -----------------------------------------------------------
votes = df[['inspector_A', 'inspector_B', 'inspector_C']].sum(axis=1)
df['label_consensus'] = (votes >= 2).astype(int)   # >=2 of 3 say defect
acc_single = (df['label_recorded'] == df['true_defect']).mean()
acc_consensus = (df['label_consensus'] == df['true_defect']).mean()
print(f'single recorded label accuracy vs truth: {acc_single:.3f}')
print(f'majority-vote consensus accuracy vs truth: {acc_consensus:.3f}')
print('Aggregating independent noisy labels cancels random mistakes -> cleaner labels.')

single recorded label accuracy vs truth: 0.777
majority-vote consensus accuracy vs truth: 0.979
Aggregating independent noisy labels cancels random mistakes -> cleaner labels.


In [ ]:
#EXERCISE 2 — Where does consensus help most?
#Count the rows where label_recorded is wrong but label_consensus is right.
#In a comment, explain why majority voting helps most on borderline parts (where one inspector slips but two agree) — and its cost (you must pay for multiple labels).

In [6]:
# 1. Find rows where the recorded label is wrong
# but the consensus label is correct

fixed_rows = df[
    (df['label_recorded'] != df['true_defect']) &
    (df['label_consensus'] == df['true_defect'])
]

print("Number of rows fixed by consensus:", len(fixed_rows))

# (Optional) Display those rows
print(fixed_rows)

# 2. Comment:
# Majority voting improves label quality because if one inspector
# makes a mistake on a borderline part, the other two inspectors
# can still agree on the correct label.
#
# This reduces random labeling errors and produces cleaner data.
# However, the cost is higher because multiple inspectors are
# required to label each sample, increasing time and expense.

Number of rows fixed by consensus: 522
      porosity_pct  wall_thickness_mm  fill_time_s  melt_temp_c  pressure_bar  \
1             1.14               5.47         2.37        706.4          85.7   
3             0.58               7.04         2.36        705.9          88.6   
4             0.55               5.59         2.39        712.8         116.6   
6             3.02               4.89         2.31        745.1          98.7   
12            4.83               5.94         2.46        726.4          87.8   
...            ...                ...          ...          ...           ...   
2379          2.44               7.71         2.59        711.0          92.0   
2385          2.06               6.31         2.41        725.5          93.0   
2387          2.61               5.32         2.39        704.9         110.0   
2390          0.99               5.36         2.29        719.3          99.1   
2399          1.77               6.24         2.34        712.1       

In [ ]:
#3. Find likely label errors — triage what to re-inspect

In [7]:
# -----------------------------------------------------------
# 🔹 3A. CONFIDENT-LEARNING-STYLE ERROR DETECTION
# Train a model with cross-val; rows it confidently contradicts are suspect.
# -----------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
X = df[feat_cols].values
y_noisy = df['label_recorded'].values
clf = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=300, random_state=0))
proba = cross_val_predict(clf, X, y_noisy, cv=5, method='predict_proba')[:, 1]
# suspect = model is confident the label is the OTHER class
suspect = ((proba > 0.80) & (y_noisy == 0)) | ((proba < 0.20) & (y_noisy == 1))
really_wrong = (df['label_recorded'] != df['true_defect']).values
base_rate = really_wrong.mean()
precision = (suspect & really_wrong).sum() / max(suspect.sum(), 1)
print(f'flagged {int(suspect.sum())} rows as suspect.')
print(f'of the flagged rows, {precision:.1%} really were wrong  (vs {base_rate:.1%} base error rate).')
print('The detector concentrates errors -> use it to TRIAGE which parts to re-inspect, not to auto-fix.')

flagged 107 rows as suspect.
of the flagged rows, 48.6% really were wrong  (vs 22.3% base error rate).
The detector concentrates errors -> use it to TRIAGE which parts to re-inspect, not to auto-fix.


In [ ]:
#EXERCISE 3 — Detection lift
#Compute how many real errors sit in the flagged set vs how many you'd expect if you picked the same number of rows at random (suspect.sum() * base_rate).
#In a comment, explain why a detector with ~2x lift is valuable even at <100% precision — it lets a limited re-inspection budget find errors far faster than checking everything.

In [8]:
# 1. Real errors found in the flagged set
real_errors_found = (suspect & really_wrong).sum()

# Expected errors if the same number of rows were chosen randomly
expected_random = suspect.sum() * base_rate

# Detection lift
lift = real_errors_found / max(expected_random, 1)

print("Real errors found:", real_errors_found)
print("Expected errors by random selection:", expected_random)
print("Detection Lift:", round(lift, 2), "x")

# 2. Comment:
# Detection lift measures how much better the detector performs
# compared to random inspection.
#
# Even if precision is less than 100%, a detector with around 2x lift
# means the same inspection budget finds about twice as many labeling
# errors as random checking. This makes the re-inspection process much
# more efficient while reducing cost and time.

Real errors found: 52
Expected errors by random selection: 23.852083333333336
Detection Lift: 2.18 x


In [ ]:
#4. The data-centric payoff — re-labeling beats re-modelling

In [9]:
# -----------------------------------------------------------
# 🔹 4A. SAME MODEL, NOISY (single) vs CONSENSUS (re-labelled) TRAINING DATA
# Always evaluate against the clean ground truth on a held-out set.
# -----------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
idx = np.arange(len(df))
tr, te = train_test_split(idx, test_size=0.3, random_state=42, stratify=df['true_defect'])
def train_eval(train_labels, model=None):
    model = model or RandomForestClassifier(n_estimators=300, random_state=0)
    m = make_pipeline(StandardScaler(), model)
    m.fit(X[tr], train_labels[tr])
    return f1_score(df['true_defect'].values[te], m.predict(X[te]))
f1_noisy = train_eval(df['label_recorded'].values)
f1_consensus = train_eval(df['label_consensus'].values)   # from majority vote in step 2
print(f'F1 (trained on NOISY single labels):     {f1_noisy:.3f}')
print(f'F1 (trained on CONSENSUS re-labelled):   {f1_consensus:.3f}')
print(f'data-centric gain from better labels:    {f1_consensus - f1_noisy:+.3f}')
print('Same model, same features — only the labels improved.')

F1 (trained on NOISY single labels):     0.504
F1 (trained on CONSENSUS re-labelled):   0.667
data-centric gain from better labels:    +0.162
Same model, same features — only the labels improved.
